In [ ]:
/_static/db/webshop_bad.db

# 2. Normalisatie

## Situatie: we starten met een “slechte” database

In deze les vertrekken we **bewust van een slecht ontworpen databank**.
Dat klinkt vreemd, maar het is precies zo dat je leert **waarom normalisatie nodig is**.

> 💡 *In het echte leven bestaan databanken vaak al. Je moet ze begrijpen en verbeteren — niet vanaf nul perfect ontwerpen.*

We starten met **één enkele tabel** waarin **alles samen zit**:

* klanten
* bestellingen
* producten
* betalingsinfo
* promoties

Dat werkt… totdat het niet meer werkt.

## Stap 0 — De startdatabase verkennen

### De tabel `orders_wide`

Voer eerst onderstaande query uit om te zien **hoe de databank eruitziet**:

In [ ]:
SELECT * FROM orders_wide;

Gebruik ook de **schema-viewer** in de cursus en bekijk:

* de kolommen
* de datatypes
* de afwezigheid van relaties of constraints

### Denkvragen (nog niet oplossen!)

Beantwoord deze vragen **in woorden**, niet met SQL:

1. Zie je gegevens die **meermaals terugkomen**?
2. Welke kolommen horen logisch gezien:

   * bij een **klant**?
   * bij een **product**?
   * bij een **bestelling**?
3. Welke fouten zouden hier makkelijk kunnen ontstaan als:

   * een e-mail verkeerd gespeld wordt?
   * een product duurder wordt?
   * een klant verhuist?

## Stap 1 — Problemen zichtbaar maken met SELECT

We gebruiken nu **SELECT-queries** om aan te tonen dat deze structuur problematisch is.

### 1.1 Klantgegevens worden herhaald

Hoe vaak komt dezelfde klantnaam voor?

In [ ]:
SELECT customer_name, COUNT(*) AS aantal_rijen
FROM orders_wide
GROUP BY customer_name;

👉 Eén klant → meerdere rijen
Dat betekent: **klantgegevens worden telkens opnieuw opgeslagen**.

### 1.2 Inconsistenties opsporen

#### Eén persoon, meerdere e-mailadressen?

In [ ]:
SELECT customer_name,
       COUNT(DISTINCT customer_email) AS aantal_emails
FROM orders_wide
GROUP BY customer_name
HAVING COUNT(DISTINCT customer_email) > 1;

**Denk even na**:

* Welke e-mail is de juiste?
* Wat gebeurt er als we een nieuwsbrief sturen?

#### Eén product, meerdere prijzen?

In [ ]:
SELECT product_sku,
       COUNT(DISTINCT unit_price) AS aantal_prijzen
FROM orders_wide
GROUP BY product_sku
HAVING COUNT(DISTINCT unit_price) > 1;

👉 Dit is een **klassieke anomalie**:

* Is de prijs veranderd?
* Of is dit een fout?
* En hoe weet je dat achteraf nog?

### Tussenbesluit

De tabel **werkt technisch**, maar:

* ze bevat **redundantie** (herhaling),
* ze laat **inconsistenties** toe,
* ze is **moeilijk te onderhouden**.

➡️ Tijd om dit stap voor stap te verbeteren.

## Stap 2 — Eerste normalisatiestap: klanten apart zetten

We beginnen met iets logisch en herkenbaar: **klanten**.

### Waarom klanten apart?

* Een klant kan **meerdere bestellingen** hebben
* Een klant heeft:

  * één naam
  * één e-mail
  * één woonplaats
* Die info hoort **niet** op elke orderregel opnieuw te staan

Dit is een eerste stap richting **normalisatie**
(je zit hier intuïtief richting 2NF / 3NF te werken).

### 2.1 Nieuwe tabel `customers` maken

In [ ]:
CREATE TABLE customers (
    customer_id     INTEGER PRIMARY KEY,
    email           TEXT UNIQUE,
    name            TEXT,
    city            TEXT,
    postcode        TEXT
);

Let op:

* `customer_id` is een **surrogaat-sleutel**
* `email` is **UNIQUE** → geen dubbele klanten

### 2.2 Klanten overzetten vanuit de slechte tabel

We halen **unieke klanten** uit `orders_wide`:

In [ ]:
INSERT INTO customers (email, name, city, postcode)
SELECT DISTINCT
    customer_email,
    customer_name,
    customer_city,
    customer_postcode
FROM orders_wide;

-- Bekijk nu het resultaat:

SELECT * FROM customers;

### Denkvragen

1. Waarom gebruiken we `SELECT DISTINCT`?
2. Wat gebeurt er als twee rijen dezelfde e-mail hebben, maar een andere naam?
3. Welke klantgegevens zijn nu **niet meer afhankelijk van bestellingen**?

## Stap 3 — Tweede normalisatiestap: producten apart zetten

We doen nu hetzelfde voor **producten**.

### Waarom producten apart?

* Een product:

  * heeft één SKU
  * één naam
* Productinformatie hoort **niet te veranderen per bestelling**
* Prijs kan later speciaal behandeld worden (daar komen we op terug)

### 3.1 Tabel `products` aanmaken

In [ ]:
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    sku          TEXT UNIQUE,
    name         TEXT
);

### 3.2 Producten migreren

In [ ]:
-- hier zit een probleem als eenzelfde SKU met verschillende namen voorkomt
INSERT INTO products (sku, name)
SELECT DISTINCT
    product_sku,
    product_name
FROM orders_wide;


-- zo kan het wel:
/*
INSERT INTO products (sku, name)
SELECT product_sku,
       MIN(product_name) AS name
FROM orders_wide
GROUP BY product_sku;
*/
-- Controleer:

SELECT * FROM products;

### Reflectie

Beantwoord deze vragen in je eigen woorden:

1. Welke gegevens **zitten nu niet meer** in `orders_wide`?
2. Waarom is het logisch dat `sku` uniek is?
3. Welke problemen zijn al opgelost, en welke **nog niet**?

> ⚠️ Let op:
> De oorspronkelijke tabel bestaat nog steeds.
> We hebben **nog niets verwijderd** — alleen voorbereid.


## Stap 4 – Bestellingen structureren

### Wat is een bestelling eigenlijk?

Een **bestelling**:

* hoort bij **één klant**
* heeft een **datum**
* kan **meerdere producten** bevatten

Dat betekent:

* bestelling ≠ product
* bestelling ≠ klant

➡️ We maken dus een **orders-tabel**.

### 4.1 Orders-tabel aanmaken

In [ ]:
CREATE TABLE orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_date TEXT NOT NULL,
    customer_id INTEGER NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

Let op:

* `customer_id` is een **foreign key**
* die verwijst naar `customers(customer_id)`, de primaire sleutel van de klantentabel

### 4.2 Orders vullen (zonder producten!)

In [ ]:
INSERT INTO orders (order_date, customer_id)
SELECT
    o.order_date,
    c.customer_id
FROM orders_wide o
JOIN customers c
    ON o.customer_email = c.email
GROUP BY o.order_id;

🧠 Denkvragen:

1. Waarom gebruiken we hier `JOIN`?
2. Waarom mogen we `product_*` hier **niet** meenemen?
3. Wat betekent `GROUP BY o.order_id` in deze context?

### Controle

In [ ]:
SELECT * FROM orders;

Elke rij stelt nu **één bestelling** voor, los van producten.

## Stap 5 – De many-to-many relatie: order_items

### Waarom is dit nodig?

Eén bestelling:

* bevat meerdere producten

Eén product:

* komt voor in meerdere bestellingen

➡️ Dat is een **many-to-many relatie**
➡️ Die lossen we op met een **koppeltabel**

### 5.1 Order items-tabel aanmaken

In [ ]:
CREATE TABLE order_items (
    order_id INTEGER NOT NULL,
    product_sku TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    price REAL NOT NULL,
    PRIMARY KEY (order_id, product_sku),
    FOREIGN KEY (order_id) REFERENCES orders(id),
    FOREIGN KEY (product_sku) REFERENCES products(sku)
);

💡 Hier gebeurt veel:

* samengestelde primaire sleutel
* twee foreign keys
* `quantity` en `price` horen **bij de combinatie**, niet bij product of order

### 5.2 Order items vullen

In [ ]:
INSERT INTO order_items (order_id, product_sku, quantity, price)
SELECT
    o.id,
    ow.product_sku,
    ow.quantity,
    ow.unit_price
FROM orders_wide ow
JOIN orders o
    ON ow.order_date = o.order_date;

⚠️ Dit werkt hier omdat:

* de oefendata vereenvoudigd is
* elke orderdatum uniek is per bestelling

🧠 Denk na:

* waarom zou dit in een echte database gevaarlijk zijn?
* wat zou een betere sleutel zijn?

### Controle

In [ ]:
SELECT * FROM order_items;

## Stap 6 – Overzicht: van chaos naar structuur

We zijn geëindigd met:

* `customers`
* `products`
* `orders`
* `order_items`

De oorspronkelijke tabel `orders_wide`:

* was handig om te starten
* maar is nu **overbodig**

### (Optioneel) Oude tabel verwijderen

In [ ]:
DROP TABLE orders_wide;

⚠️ In echte systemen doe je dit pas:

* na testen
* na back-ups
* na akkoord van stakeholders

## Tot slot

> Normalisatie is geen trucje
> Het is een **manier van denken**

Elke stap:

* verminderde redundantie
* maakte fouten zichtbaarder
* dwong expliciete keuzes af